# Removal checks: Sec. III worked through

Companion to `../../Manuscript/main.tex` (the paper repository), Sec. III
("Removal: fluctuations, intermittency and the limits of closure"). This is
a trimmed version of a longer didactic walkthrough: it keeps only the parts
that back the "Numerical checks" paragraph of Sec. S4 of the Supplemental
Material (plateau variance under continuous removal, and the sloughing
variance), plus the moment check the rest of the notebook depends on.

Dropped relative to the original walkthrough, because each is superseded by
a notebook already in this repository or is not part of the public
reproducibility story:

- the old Fig. 4/`fig_removal` assembly — superseded by `01_figures.ipynb`
  (Fig. 3 of the current manuscript);
- the nonlinear-removal closure demo — superseded by `01_figures.ipynb`'s
  Fig. S3 cell, which already reproduces the $2.90/2.79/2.80\pm0.01$ numbers;
- the industrial scale-separation section — it validated the derivation
  against a proprietary dataset under a confidentiality agreement, which is
  not reproducible from this public repository and is therefore out of
  scope here;
- the Kern–Seaton restatement — superseded by `01_figures.ipynb`'s Fig. 3;
- the first-passage/remaining-useful-life section — superseded by
  `02_first_passage_validation.ipynb`.

Everything uses the three matched members of the tempered-stable family at a
common $(m,c,\tau_0)$, as in Sec. II.

## Setup: the three matched deposition laws

Every law in this notebook is one of the three members of the
tempered-stable family (Sec. II, Eq. `tempered_stable`) matched to a common
$(m,c,\tau_0)$: compound Poisson ($\alpha=-1$), Gamma ($\alpha=0$), inverse
Gaussian ($\alpha=1/2$). The exact increment laws at any window $\Delta$
are given in Appendix A.2 ("Tempered-stable family: derivations"):

- **CPP**: $N\sim\mathrm{Poisson}(r\Delta)$ jumps, each
  $\mathrm{Exp}(\mathrm{mean}=\ell/2)$, $r=2/t_c$.
- **Gamma**: $\Delta D\sim\mathrm{Gamma}(\mathrm{shape}=\Delta/t_c,\ \mathrm{scale}=\ell)$.
- **Inverse Gaussian**: $\Delta D\sim\mathrm{IG}(\mu_\Delta=m\Delta,\ \lambda_{\mathrm{IG}}=m\Delta^2/t_c)$,
  sampled with the exact Michael-Schucany-Haas (1976) algorithm.

with $\ell=c^2m\tau_0$ (size-biased event size) and $t_c=\ell/m$.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
plt.rcParams.update({"font.size": 9, "figure.dpi": 130, "pdf.fonttype": 42, "ps.fonttype": 42})

LAWS = ["cpp", "gamma", "ig"]
LAW_LABEL = {"cpp": "compound Poisson", "gamma": "Gamma", "ig": "inverse Gaussian"}
LAW_COLOR = {"cpp": "#1b9e77", "gamma": "#d95f02", "ig": "#7570b3"}


def ell_tc(m, c, tau0):
    """Size-biased event size ell and mean inter-event time t_c (Sec. II)."""
    ell = c**2 * m * tau0
    return ell, ell / m


def sample_ig(mu, lam, size, rng):
    """Exact IG(mu, lam) sampler (Michael, Schucany & Haas, 1976)."""
    mu = np.broadcast_to(mu, (size,)).astype(float)
    lam = np.broadcast_to(lam, (size,)).astype(float)
    y = rng.standard_normal(size) ** 2
    x = mu + mu**2 * y / (2 * lam) - (mu / (2 * lam)) * np.sqrt(
        np.clip(4 * mu * lam * y + mu**2 * y**2, 0, None))
    z = rng.uniform(size=size)
    return np.where(z <= mu / (mu + x), x, mu**2 / x)


def sample_increment(law, m, c, tau0, Delta, size, rng):
    """Exact sample of D(t+Delta)-D(t) for the matched tempered-stable laws."""
    ell, tc = ell_tc(m, c, tau0)
    if law == "cpp":
        lam, r = 2.0 / ell, 2.0 / tc
        N = rng.poisson(r * Delta, size=size)
        out = np.zeros(size)
        maxN = int(N.max()) if N.size else 0
        if maxN > 0:
            jumps = rng.exponential(scale=ell / 2.0, size=(size, maxN))
            mask = np.arange(maxN)[None, :] < N[:, None]
            out = (jumps * mask).sum(axis=1)
        return out
    elif law == "gamma":
        return rng.gamma(shape=Delta / tc, scale=ell, size=size)
    elif law == "ig":
        mu, lam = m * Delta, m * Delta**2 / tc
        return sample_ig(mu, lam, size, rng)
    raise ValueError(law)


**Moment check**, against Eqs. `deposition_mean`, `deposition_variance`
and the tempered-stable skewness Eq. `ts_skewness`
($\gamma_1=\tfrac{2-\alpha}{1-\alpha}c$):

In [2]:
m, c, tau0, Delta = 1.0, 2.0, 1.0, 1.0
ALPHA_OF = {"cpp": -1.0, "gamma": 0.0, "ig": 0.5}
N = 300_000
for law in LAWS:
    x = sample_increment(law, m, c, tau0, Delta, N, rng)
    alpha = ALPHA_OF[law]
    gamma1_theory = (2 - alpha) / (1 - alpha) * c
    gamma1_mc = ((x - x.mean())**3).mean() / x.std()**3
    print(f"{law:8s}  mean {x.mean():.3f} (target {m*Delta:.3f})   "
          f"var {x.var():.3f} (target {c**2*m**2*tau0*Delta:.3f})   "
          f"skew {gamma1_mc:.2f} (target {gamma1_theory:.2f})")


cpp       mean 0.998 (target 1.000)   var 3.995 (target 4.000)   skew 3.02 (target 3.00)
gamma     mean 0.997 (target 1.000)   var 3.964 (target 4.000)   skew 3.96 (target 4.00)
ig        mean 1.001 (target 1.000)   var 4.023 (target 4.000)   skew 5.93 (target 6.00)


## Sec. III.A — Linear removal: a Lévy-driven Ornstein–Uhlenbeck process

Solving $dR=dD-\psi R\,dt$ gives the shot-noise/OU representation
(Eq. `ou_solution`). We simulate it with the standard discretisation
$R_{k+1}=R_k e^{-\psi\,dt}+\Delta D_k$, and check the stationary plateau
against Eq. `plateau`:
$$R_\infty=\frac{m}{\psi},\qquad \mathrm{Var}[R]_\infty=\frac{s^2}{2\psi}.$$

In [3]:
def simulate_ou(law, m, c, tau0, psi, T, dt, n_paths, rng, R0=0.0):
    n_steps = int(round(T / dt))
    R = np.full(n_paths, R0, dtype=float)
    decay = np.exp(-psi * dt)
    path = np.empty((n_paths, n_steps + 1))
    path[:, 0] = R
    for k in range(n_steps):
        R = R * decay + sample_increment(law, m, c, tau0, dt, n_paths, rng)
        path[:, k + 1] = R
    return path


m, c, tau0, psi = 1.0, 1.0, 1.0, 0.2
R_inf, s2 = m / psi, c**2 * m**2 * tau0
T, dt = 60.0, 0.05

tails = {}
for law in LAWS:
    paths = simulate_ou(law, m, c, tau0, psi, T, dt, 4000, rng)
    tails[law] = paths[:, -1]
    print(f"{law:8s}  plateau mean {tails[law].mean():.3f} (target {R_inf:.3f})   "
          f"plateau var {tails[law].var():.3f} (target {s2/(2*psi):.3f})")


cpp       plateau mean 4.993 (target 5.000)   plateau var 2.468 (target 2.500)
gamma     plateau mean 5.052 (target 5.000)   plateau var 2.570 (target 2.500)


ig        plateau mean 4.971 (target 5.000)   plateau var 2.361 (target 2.500)


## Sec. III.B — Intermittent removal: sloughing

Replace continuous removal by Poisson($\rho$) events that each strip a
fixed fraction $\eta$ of the deposit (Eq. `sloughing`). The mean is
unchanged ($\psi=r_s\eta$), but the stationary variance is not
(Eq. `sloughing_variance`):
$$\mathrm{Var}[R]_\infty=\frac{s^2}{(2-\eta)\psi}+\frac{\eta}{2-\eta}R_\infty^2.$$

In [4]:
def simulate_sloughing(law, m, c, tau0, psi_target, eta, T, dt, n_paths, rng, R0=0.0):
    rho = psi_target / eta
    n_steps = int(round(T / dt))
    R = np.full(n_paths, R0, dtype=float)
    path = np.empty((n_paths, n_steps + 1))
    path[:, 0] = R
    p_event = 1 - np.exp(-rho * dt)
    for k in range(n_steps):
        R = R + sample_increment(law, m, c, tau0, dt, n_paths, rng)
        fire = rng.uniform(size=n_paths) < p_event
        R = np.where(fire, R * (1 - eta), R)
        path[:, k + 1] = R
    return path


for eta in [0.2, 0.5, 0.8, 1.0]:
    paths = simulate_sloughing("gamma", m, c, tau0, psi, eta, T, dt, 3000, rng)
    analytic = s2 / ((2 - eta) * psi) + eta / (2 - eta) * R_inf**2
    print(f"eta={eta:.1f}  MC var {paths[:, -1].var():6.2f}   analytic {analytic:6.2f}")


eta=0.2  MC var   5.30   analytic   5.56


eta=0.5  MC var  12.02   analytic  11.67


eta=0.8  MC var  21.94   analytic  20.83


eta=1.0  MC var  27.83   analytic  30.00


## What maps to what

| This notebook | Paper object |
|---|---|
| Moment check | Table II of the main text |
| OU plateau check | Sec. III.A, plateau relations (Table I of the main text) |
| Sloughing variance check | Sec. III.B, sloughing variance (Table I); backs the $\eta=0.5$, Gamma-deposition number in the Supplemental Material's "Numerical checks" paragraph (Sec. S4) |

Seed fixed at the top (`rng = np.random.default_rng(42)`); runtime well
under a minute for all cells above.